# Logistic Regression with GridSearchCV on a Balanced (Undersampled) Training Set

**Project:** Machine-learning-based recognition of production and changeover activities  
**CRISP-DM phase:** Modelling / Evaluation

## Purpose

About 70 % of the rows are Changeover (0) and 30 % are Production (1). This notebook tests whether **balancing the classes** helps a logistic regression for `Label_03` (**1 = Production, 0 = Changeover**). It uses the *original, unscaled* dataset and the analog-like signals only. It:

1. keeps numeric features with more than 20 distinct values (roughly the analog signals) and removes all label columns, `Stuckzahl` and `time`;
2. makes a stratified random 80/20 train/test split;
3. balances the **training set only** to 50/50 by randomly removing rows of the majority class (undersampling);
4. runs `GridSearchCV` on the balanced training set;
5. evaluates the best model on the untouched, imbalanced test set.

> **Data:** The dataset is public (Zenodo, licence CC BY 4.0) and is **not** included in this repository. See the project README for the download link and citation. This notebook reads the original dataset file, saved as `data.csv` in the project root. Generated files are written to `../outputs/` and are not committed.

## Important limitations

- **Random split on time-series data.** The split is a stratified *random* split, not a chronological one. Neighbouring time steps are very similar, so scores can be optimistic.
- **Undersampling discards data.** Many majority-class rows are not used for training. Precision on the imbalanced test set is lower than on a balanced one because Changeover rows are more frequent there.
- **No scaling by notebook 01.** The raw signals are standardized inside the pipeline only.

## 1. Environment and configuration

Install the dependencies with `pip install -r requirements.txt` (run from the project root). All paths are relative to this notebook, which is expected to live in the `notebooks/` folder.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 50)

DATA_PATH = Path("../data.csv")     # original dataset, see the project README
TARGET_COL = "Label_03"
MIN_UNIQUE_VALUES = 20              # keep only analog-like signals
TEST_SIZE = 0.2
SEED = 42

## 2. Load the original data

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.resolve()}. "
        "Download data_v1_0_1.csv from https://doi.org/10.5281/zenodo.15735480, "
        "save it as data.csv in the project root, or update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

if TARGET_COL not in df.columns:
    raise KeyError(f"Target column {TARGET_COL!r} not found in the data.")

## 3. Build the feature matrix and target (analog signals only)

Only numeric columns are used. All other label columns, `time` and `Stuckzahl` are removed, and the target is explicitly excluded from the features. Binary and few-valued signals (20 distinct values or fewer) are dropped.

In [ ]:
num_df = df.select_dtypes(include=[np.number]).copy()
num_df[TARGET_COL] = df[TARGET_COL].astype(int)

label_cols = [c for c in df.columns if c.startswith("Label_")]
drop_cols = [c for c in label_cols if c != TARGET_COL]
for col in ["time", "Stuckzahl"]:
    if col in num_df.columns and col not in drop_cols:
        drop_cols.append(col)

n_unique = num_df.nunique()
analog_features = [
    c for c in num_df.columns
    if c not in drop_cols and c != TARGET_COL and n_unique[c] > MIN_UNIQUE_VALUES
]

X = num_df[analog_features]
y = num_df[TARGET_COL].astype(int).to_numpy()

assert TARGET_COL not in X.columns, "The target must not be part of the features."
assert not any(c.startswith("Label_") for c in X.columns), "No label column may be a feature."

print(f"Numeric columns: {num_df.shape[1]}")
print(f"Dropped label / time / id columns: {drop_cols}")
print(f"Analog-like features kept (more than {MIN_UNIQUE_VALUES} distinct values): {len(analog_features)}")
print(f"Class counts (0 = Changeover, 1 = Production): {np.bincount(y).tolist()}")

## 4. Stratified 80/20 train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED
)

print(f"Train size: {len(X_train):,}   class counts: {np.bincount(y_train).tolist()}")
print(f"Test size:  {len(X_test):,}   class counts: {np.bincount(y_test).tolist()}")

## 5. Random undersampling of the majority class (training set only)

The test set is never touched, so the evaluation still reflects the real class balance.

In [ ]:
rng = np.random.RandomState(SEED)

idx_class0 = np.where(y_train == 0)[0]
idx_class1 = np.where(y_train == 1)[0]

n_min = min(len(idx_class0), len(idx_class1))
idx0_down = rng.choice(idx_class0, size=n_min, replace=False)
idx1_down = rng.choice(idx_class1, size=n_min, replace=False)

undersampled_idx = np.concatenate([idx0_down, idx1_down])
rng.shuffle(undersampled_idx)

X_train_bal = X_train.iloc[undersampled_idx]
y_train_bal = y_train[undersampled_idx]

print(f"Balanced train size: {len(X_train_bal):,}")
print(f"Balanced class counts: {np.bincount(y_train_bal).tolist()}")

## 6. Grid search on the balanced training set

The pipeline imputes, standardizes and fits a logistic regression without further class weighting (the data is already balanced). `GridSearchCV` tries 4 values of `C` and 2 penalties with 5-fold cross-validation and selects by F1-score.

In [ ]:
pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight=None,       # no extra weighting: the data is already balanced
        solver="liblinear",
    )),
])

param_grid = {
    "model__C": [0.01, 0.1, 1.0, 10.0],
    "model__penalty": ["l1", "l2"],
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
)

print("Running GridSearchCV on the balanced training set...")
grid.fit(X_train_bal, y_train_bal)

print("Best parameters:", grid.best_params_)
print(f"Best cross-validated F1-score (balanced training data): {grid.best_score_:.4f}")

best_model = grid.best_estimator_

## 7. Evaluation on the original, imbalanced test set

In [ ]:
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print(f"Test accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Test precision: {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Test recall:    {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Test F1-score:  {f1_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Test ROC-AUC:   {roc_auc_score(y_test, y_proba):.4f}")

print("\nClassification report:")
print(classification_report(y_test, y_pred, digits=4))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(4, 3))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Pred 0", "Pred 1"], yticklabels=["True 0", "True 1"])
plt.title("Confusion matrix: LR GridSearch (undersampled)")
plt.tight_layout()
plt.show()

## 8. Summary and conclusion

Balancing the training data makes the model treat Production and Changeover equally. In the original run this gave a high recall for Production, at the price of more false alarms (lower precision), compared with the 40-feature baseline of notebook 02.

**Reference results from the author's original run** (84 analog-like features; best parameters `C = 0.1` with an L1 penalty; best cross-validated F1 on the balanced training data 0.8631). The cleaned notebook may give slightly different numbers when re-run:

| Test metric (imbalanced test set) | Value |
|---|---|
| AUC | 0.9236 |
| Accuracy | 0.8459 |
| F1 | 0.7754 |
| Precision (Production) | 0.6910 |
| Recall (Production) | 0.8832 |

**Next steps:** compare with a chronological split, and with class weighting instead of undersampling (`class_weight="balanced"` keeps all rows).